# LeetCode 418 — Sentence Screen Fitting bằng Python

Link đề: https://leetcode.com/problems/sentence-screen-fitting/ · Độ khó: **Medium** · Chủ đề: Array, String, Dynamic Programming, Simulation

Skeleton LeetCode:

```python
class Solution:
    def wordsTyping(self, sentence: List[str], rows: int, cols: int) -> int:
        ...
```

Ghi chú: đây là bài **Premium** (khóa). Phần phát biểu đề dưới đây được viết lại theo hiểu biết về bài và theo đề gốc đã cung cấp; nếu bạn có tài khoản Premium, hãy đối chiếu lại đề gốc trên LeetCode. Các ví dụ đều được chạy lại bằng code trong notebook.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho màn hình `rows x cols` ký tự và một câu gồm các từ; viết câu lặp đi lặp lại, từ không bị cắt đôi, giữa hai từ liền nhau trên cùng một dòng có đúng **một dấu cách**; đếm xem câu được viết trọn vẹn bao nhiêu lần.
2. Nắm công cụ Python cần dùng: `" ".join`, `len`, phép `%` và `//` để "quay vòng" trên một chuỗi lặp, vòng `while` có điều kiện dừng, `dict` để nhớ trạng thái đã gặp.
3. Đi từ **mô phỏng từng từ (brute force)** → **nối cả câu thành một chuỗi vòng tròn có dấu cách ở cuối rồi nhảy từng dòng** → **tính trước "dòng bắt đầu bằng từ i thì kết thúc ở từ nào"** → **phát hiện chu kỳ để nhảy cóc**.
4. Hiểu mấu chốt: *dấu cách cuối câu* biến luật "giữa hai từ có một dấu cách" thành luật đồng nhất, và việc "lùi về dấu cách gần nhất" chính là luật "không cắt đôi từ".
5. Tránh các lỗi kinh điển: quên dấu cách cuối câu, kiểm tra sai chỉ số khi lùi (`s[cur]` thay vì `s[cur-1]`), vòng lặp vô hạn khi có từ dài hơn `cols`, đếm sai khi dòng kết thúc đúng tại ranh giới câu.

## 2. Đề bài & ví dụ

Cho một màn hình có `rows` dòng, mỗi dòng `cols` ô ký tự, và một câu `sentence` được biểu diễn dưới dạng **danh sách các từ**. Ta viết câu lên màn hình **lặp lại nhiều lần** theo luật:

- Thứ tự các từ **không đổi** (hết từ cuối lại quay về từ đầu).
- Một từ **không được tách** thành hai dòng.
- Hai từ liên tiếp **trên cùng một dòng** cách nhau **đúng một dấu cách**.

Trả về **số lần** câu đã được viết **trọn vẹn** lên màn hình.

**Ví dụ 1:** `sentence = ["hello","world"], rows = 2, cols = 8` → `1`.

```
hello---
world---
```
(`-` là ô trống.) Dòng 1 chứa `hello` (5 ô), còn 3 ô nhưng `world` cần thêm dấu cách + 5 ô = 6 ô nên xuống dòng.

**Ví dụ 2:** `sentence = ["a","bcd","e"], rows = 3, cols = 6` → `2`.

```
a-bcd-
e-a---
bcd-e-
```

**Ví dụ 3:** `sentence = ["i","had","apple","pie"], rows = 4, cols = 5` → `1`.

```
i-had
apple
pie-i
had--
```

**Ràng buộc:** `1 <= sentence.length <= 100`, `1 <= sentence[i].length <= 10`, `sentence[i]` gồm chữ thường, `1 <= rows, cols <= 2 * 10^4`.

In [1]:
from typing import List
import random
import time

examples = [
    (["hello", "world"], 2, 8, 1),
    (["a", "bcd", "e"], 3, 6, 2),
    (["i", "had", "apple", "pie"], 4, 5, 1),
]
for sentence, rows, cols, expected in examples:
    print(sentence, "rows =", rows, "cols =", cols, "-> kỳ vọng", expected)

['hello', 'world'] rows = 2 cols = 8 -> kỳ vọng 1
['a', 'bcd', 'e'] rows = 3 cols = 6 -> kỳ vọng 2
['i', 'had', 'apple', 'pie'] rows = 4 cols = 5 -> kỳ vọng 1


### 2.1. Đọc kỹ đề: vài điểm dễ hiểu nhầm

- **"Số lần câu được viết trọn vẹn"** — phần dở dang ở cuối màn hình **không** được tính. Ở ví dụ 3 màn hình chứa `i had apple pie i had`, câu đầy đủ chỉ xuất hiện 1 lần (lần thứ hai mới có `i had`).
- **Dấu cách giữa hai câu liên tiếp:** sau từ cuối (`pie`) rồi lại tới từ đầu (`i`) trên cùng một dòng thì vẫn phải cách nhau **một dấu cách** (xem dòng `pie-i` ở ví dụ 3). Chính chi tiết này khiến ta muốn coi "dấu cách" là một phần của câu.
- **Từ dài hơn `cols`:** nếu có một từ có độ dài lớn hơn `cols` thì không thể viết được từ đó ⇒ đáp án là `0`. Với ràng buộc `len(word) <= 10` và `cols >= 1`, tình huống này **có thể xảy ra** (ví dụ `cols = 3` mà từ dài 5).
- **Kích thước:** `rows, cols` tới `2 * 10^4` ⇒ màn hình có tới `4 * 10^8` ô. Không thể "vẽ" màn hình rồi đếm; ta phải tính toán thông minh.

In [2]:
# Từ dài hơn cols thì không thể viết:
sentence, rows, cols = ["abcde", "f"], 5, 3
print([len(w) for w in sentence], "cols =", cols, "-> không từ nào 'abcde' vừa dòng, đáp án phải là 0")
print("số ô của màn hình lớn nhất:", 2 * 10**4 * 2 * 10**4)

[5, 1] cols = 3 -> không từ nào 'abcde' vừa dòng, đáp án phải là 0
số ô của màn hình lớn nhất: 400000000


## 3. Kiến thức nền

### 3.1. `" ".join(...)` và `len` — nối các từ thành một chuỗi

`" ".join(words)` nối các phần tử của danh sách bằng đúng một dấu cách **ở giữa** (không có ở đầu/cuối). Nếu muốn có dấu cách ở cuối, ta cộng thêm `" "`.

In [3]:
words = ["a", "bcd", "e"]
s1 = " ".join(words)
s2 = " ".join(words) + " "
print(repr(s1), len(s1))      # 'a bcd e'  độ dài 7
print(repr(s2), len(s2))      # 'a bcd e ' độ dài 8 (có dấu cách cuối)

'a bcd e' 7
'a bcd e ' 8


### 3.2. Phép `%` — quay vòng chỉ số trên chuỗi lặp

Nếu `s` có độ dài `L` và ta lặp `s` vô hạn lần, thì ký tự ở vị trí `p` của chuỗi vô hạn đó chính là `s[p % L]`. Ta **không** cần tạo chuỗi dài, chỉ cần `%`.

In [4]:
s = "a bcd e "                # L = 8
L = len(s)
infinite = s * 3              # chuỗi lặp 3 lần để đối chiếu
for p in [0, 7, 8, 13, 20]:
    print(f"p={p:>2}: s[p % L] = {s[p % L]!r}  | infinite[p] = {infinite[p]!r}")

p= 0: s[p % L] = 'a'  | infinite[p] = 'a'
p= 7: s[p % L] = ' '  | infinite[p] = ' '
p= 8: s[p % L] = 'a'  | infinite[p] = 'a'
p=13: s[p % L] = ' '  | infinite[p] = ' '
p=20: s[p % L] = 'd'  | infinite[p] = 'd'


### 3.3. `//` — chia lấy phần nguyên: đếm số vòng trọn vẹn

`p // L` cho biết ta đã đi qua trọn vẹn bao nhiêu lần chuỗi `s`. Ví dụ đã "phủ" được `p = 17` ký tự của chuỗi vô hạn thì có `17 // 8 = 2` câu trọn vẹn (còn dư `17 % 8 = 1` ký tự của câu thứ ba).

In [5]:
L = 8
for p in [0, 7, 8, 16, 17]:
    print(f"p={p:>2}: p // L = {p // L} câu trọn vẹn, dư p % L = {p % L}")

p= 0: p // L = 0 câu trọn vẹn, dư p % L = 0
p= 7: p // L = 0 câu trọn vẹn, dư p % L = 7
p= 8: p // L = 1 câu trọn vẹn, dư p % L = 0
p=16: p // L = 2 câu trọn vẹn, dư p % L = 0
p=17: p // L = 2 câu trọn vẹn, dư p % L = 1


### 3.4. Vòng `while` có điều kiện dừng — "lùi cho đến khi gặp dấu cách"

Ta sẽ cần dạng: *chừng nào ký tự ngay trước vị trí `cur` chưa phải dấu cách thì lùi `cur` đi một*. Để ý ta kiểm tra ký tự **trước** `cur` (`cur - 1`), không phải ký tự tại `cur`.

In [6]:
s = "a bcd e "
cur = 4                       # s[4] = 'd'
print("s[cur] =", repr(s[cur]), "| s[cur-1] =", repr(s[cur - 1]))
while cur > 0 and s[(cur - 1) % len(s)] != " ":
    cur -= 1
    print("lùi về", cur)
print("dừng ở cur =", cur, "- ngay sau dấu cách, đầu từ 'bcd' (s[cur] =", repr(s[cur]), ")")

s[cur] = 'd' | s[cur-1] = 'c'
lùi về 3
lùi về 2
dừng ở cur = 2 - ngay sau dấu cách, đầu từ 'bcd' (s[cur] = 'b' )


### 3.5. `dict` để nhớ "tôi đã từng ở trạng thái này chưa?"

`seen = {}` lưu `seen[key] = value`. Kiểm tra `key in seen` mất `O(1)`. Ta dùng nó ở Cách 4 để phát hiện chu kỳ.

In [7]:
seen = {}
seen[3] = (0, 0)             # trạng thái 3 xuất hiện lần đầu ở dòng 0 với tổng 0
print(3 in seen, 5 in seen)  # True False
print(seen[3])

True False
(0, 0)


## 4. Ý tưởng nền tảng: tính tay một ví dụ

Lấy ví dụ 2: `sentence = ["a","bcd","e"]`, `cols = 6`. Hãy tự "đánh máy" từng dòng bằng tay:

- **Dòng 1:** `a` (còn 5 ô) → cần dấu cách + `bcd` = 4 ô → còn 1 ô → `e` cần 2 ô (dấu cách + `e`) nhưng chỉ còn 1 ô ⇒ dừng. Dòng 1 = `a bcd`, hết **2 từ + `bcd`** (xong từ `bcd`, từ kế tiếp là `e`).
- **Dòng 2:** `e` (còn 5 ô) → dấu cách + `a` → còn 3 ô → dấu cách + `bcd` cần 4 ô ⇒ không vừa. Dòng 2 = `e a`. Câu đã xong **1 lần** (từ `e` là từ cuối) và đang ở từ `bcd`.
- **Dòng 3:** `bcd` (còn 3 ô) → dấu cách + `e` = 2 ô, vừa ⇒ `bcd e`. Câu xong **lần 2**.

Vậy đáp án `2`. Nhận xét quan trọng: **mỗi dòng chỉ phụ thuộc vào "từ nào bắt đầu dòng"**. Đây là chìa khóa của Cách 3 và Cách 4.

In [8]:
sentence, cols = ["a", "bcd", "e"], 6
n = len(sentence)
i = 0                                   # chỉ số từ bắt đầu dòng
for row in range(1, 4):
    line = []
    used = 0
    while True:
        w = sentence[i]
        need = len(w) if not line else len(w) + 1      # từ đầu dòng không cần dấu cách phía trước
        if used + need > cols:
            break
        line.append(w)
        used += need
        i = (i + 1) % n
    print(f"dòng {row}: {' '.join(line)!r:<10} dùng {used} ô, từ bắt đầu dòng kế: {sentence[i]!r}")

dòng 1: 'a bcd'    dùng 5 ô, từ bắt đầu dòng kế: 'e'
dòng 2: 'e a'      dùng 3 ô, từ bắt đầu dòng kế: 'bcd'
dòng 3: 'bcd e'    dùng 5 ô, từ bắt đầu dòng kế: 'a'


## 5. Cách 1 — Brute Force: mô phỏng từng từ

**Ý tưởng:** viết đúng như người đánh máy. Giữ chỉ số từ hiện tại `i` và số ô còn trống `remain` của dòng đang viết. Với mỗi dòng:

1. Đặt `remain = cols`.
2. Chừng nào từ `sentence[i]` vừa (`len <= remain`): trừ `len + 1` (từ + một dấu cách phía sau), sang từ kế; nếu qua hết từ cuối thì cộng 1 vào số câu.
3. Hết chỗ thì xuống dòng.

**Dấu cách phía sau:** ta luôn trừ cả dấu cách. Ô cuối dòng không thật sự cần dấu cách, nhưng không sao: nếu `remain` thành `-1`, không từ nào (độ dài `>= 1`) vừa nữa. Nhờ vậy điều kiện `len <= remain` vẫn đúng.

**Bảo vệ vô hạn:** nếu có từ dài hơn `cols` thì vòng `while` không bao giờ nhích được `i`; ta phải trả `0` ngay từ đầu.

In [9]:
def words_typing_brute(sentence, rows, cols):
    if any(len(w) > cols for w in sentence):
        return 0                       # một từ không bao giờ vừa dòng
    n = len(sentence)
    i = 0
    count = 0
    for _ in range(rows):
        remain = cols
        while len(sentence[i]) <= remain:
            remain -= len(sentence[i]) + 1      # từ + dấu cách
            i += 1
            if i == n:
                i = 0
                count += 1
    return count


for sentence, rows, cols, expected in examples:
    got = words_typing_brute(sentence, rows, cols)
    print(sentence, rows, cols, "->", got, "| kỳ vọng", expected)
    assert got == expected

['hello', 'world'] 2 8 -> 1 | kỳ vọng 1
['a', 'bcd', 'e'] 3 6 -> 2 | kỳ vọng 2
['i', 'had', 'apple', 'pie'] 4 5 -> 1 | kỳ vọng 1


**Độ phức tạp:** mỗi từ được đặt tốn `O(1)`. Số từ được đặt khoảng `rows * cols / (độ dài trung bình + 1)`. Trường hợp xấu: toàn từ 1 chữ cái, `cols = rows = 2 * 10^4` ⇒ khoảng `2 * 10^8` từ. **Time `O(rows * cols)` trong xấu nhất**, Space `O(1)`.

**Nút thắt:** ta đặt **từng từ một** dù mỗi dòng chỉ phụ thuộc vào từ bắt đầu dòng. Hãy đo thử trên input lớn để thấy sự chậm chạp.

In [10]:
big = ["a"] * 100
t = time.perf_counter()
print(words_typing_brute(big, 2000, 2000))
print("brute rows=cols=2000: %.2f giây (với 2*10^4 sẽ gấp ~100 lần)" % (time.perf_counter() - t))

20000
brute rows=cols=2000: 0.14 giây (với 2*10^4 sẽ gấp ~100 lần)


## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát giúp đi nhanh hơn:

**Quan sát A — Dấu cách cuối câu.** Nối `s = " ".join(sentence) + " "`. Chuỗi vô hạn `s s s s ...` chính là nội dung "đánh máy liên tục không xuống dòng", trong đó mỗi từ được theo sau bởi một dấu cách (kể cả từ cuối câu, vì sau nó là dấu cách rồi tới câu kế). Độ dài `L = len(s)`.

**Quan sát B — Mỗi dòng = một bước nhảy `cols`.** Gọi `cur` là số ký tự của chuỗi vô hạn đã được "phủ" xong. Viết thêm một dòng:

1. Nhảy `cur += cols`: ta đứng ở ký tự `s[cur % L]`.
2. Nếu ký tự đó là **dấu cách** ⇒ dòng kết thúc đúng ở ranh giới từ ⇒ bỏ qua dấu cách đó (`cur += 1`), dòng kế bắt đầu bằng từ mới.
3. Ngược lại ta đang ở **giữa một từ** ⇒ từ đó không vừa dòng, phải đẩy xuống dòng sau ⇒ **lùi** `cur` cho tới khi ký tự ngay trước `cur` là dấu cách (tức `cur` đứng ở đầu từ).

Cuối cùng đáp án là `cur // L` — số lần chuỗi `s` được phủ trọn.

Hãy tự kiểm chứng bằng ví dụ 2: `s = "a bcd e "`, `L = 8`, `cols = 6`.

In [11]:
s = " ".join(["a", "bcd", "e"]) + " "
L, cols = len(s), 6
cur = 0
for row in range(1, 4):
    cur += cols
    note = ""
    if s[cur % L] == " ":
        cur += 1
        note = "rơi vào dấu cách -> +1"
    else:
        while cur > 0 and s[(cur - 1) % L] != " ":
            cur -= 1
        note = "rơi giữa từ -> lùi về đầu từ"
    print(f"sau dòng {row}: cur = {cur:>2}  ({note})")
print("đáp án =", cur // L)        # 16 // 8 = 2

sau dòng 1: cur =  6  (rơi giữa từ -> lùi về đầu từ)
sau dòng 2: cur = 10  (rơi giữa từ -> lùi về đầu từ)
sau dòng 3: cur = 16  (rơi giữa từ -> lùi về đầu từ)
đáp án = 2


Vì sao ổn? Bước "lùi" đảm bảo phần đã phủ **luôn kết thúc ngay sau một dấu cách**, tức là ngay sau một từ trọn vẹn, và dấu cách đó là khoảng cách bắt buộc giữa các từ (hoặc là ô trống cuối dòng). Ta sẽ chứng minh kỹ ở mục 10.

## 7. Các cách tiếp theo

### Cách 2 — Chuỗi vòng tròn + nhảy `cols` mỗi dòng (`O(rows * độ dài từ)`)

**Giải thích từng đoạn:**

1. `s = " ".join(sentence) + " "`, `L = len(s)`.
2. `cur = 0`. Lặp `rows` lần: `cur += cols`.
3. Nếu `s[cur % L] == " "`: `cur += 1`. Ngược lại: `while cur > 0 and s[(cur - 1) % L] != " ": cur -= 1`.
4. Trả `cur // L`.

Mỗi dòng lùi tối đa `max(len(word)) <= 10` bước ⇒ tổng `O(rows * 10)`.

In [12]:
def words_typing_joined(sentence, rows, cols):
    s = " ".join(sentence) + " "
    L = len(s)
    cur = 0
    for _ in range(rows):
        cur += cols
        if s[cur % L] == " ":
            cur += 1
        else:
            while cur > 0 and s[(cur - 1) % L] != " ":
                cur -= 1
    return cur // L


for sentence, rows, cols, expected in examples:
    assert words_typing_joined(sentence, rows, cols) == expected
print("Cách 2 đúng trên các ví dụ đề")

Cách 2 đúng trên các ví dụ đề


In [13]:
# Code nộp LeetCode — Cách 2
class Solution2:
    def wordsTyping(self, sentence: List[str], rows: int, cols: int) -> int:
        s = " ".join(sentence) + " "
        L = len(s)
        cur = 0
        for _ in range(rows):
            cur += cols
            if s[cur % L] == " ":
                cur += 1
            else:
                while cur > 0 and s[(cur - 1) % L] != " ":
                    cur -= 1
        return cur // L


print(Solution2().wordsTyping(["a", "bcd", "e"], 3, 6))   # 2

2


**Độ phức tạp:** Time `O(rows * m)` với `m` là độ dài từ lớn nhất (`<= 10`) ⇒ coi như `O(rows)`; Space `O(L)` với `L <= 100 * 11`.

**Chú ý:** vì `cur` chỉ tăng thêm tối đa `cols + 1` mỗi dòng và có thể rất lớn khi `rows = 2 * 10^4`, Python không lo tràn số; ở Java/C++ vẫn an toàn vì `2 * 10^4 * 2 * 10^4 = 4 * 10^8 < 2^31`.

### Cách 3 — Tính trước chuyển trạng thái của một dòng (DP theo từ bắt đầu dòng)

**Ý tưởng khác hẳn:** chỉ có `n <= 100` khả năng "từ nào bắt đầu dòng". Với mỗi `i`, ta mô phỏng **một dòng** bắt đầu từ từ `i` một lần duy nhất để biết:

- `nxt[i]` = chỉ số từ bắt đầu dòng kế tiếp;
- `cnt[i]` = số lần câu được hoàn thành trong dòng này (đi qua cuối câu bao nhiêu lần).

Sau đó mô phỏng `rows` dòng bằng cách tra bảng: `ans += cnt[i]; i = nxt[i]`.

**Giải thích từng đoạn:**

1. Nếu có từ dài hơn `cols` ⇒ trả `0`.
2. Với mỗi `i` trong `range(n)`: đặt `j = i`, `remain = cols`, `done = 0`; lặp `while len(sentence[j]) <= remain` như Cách 1; gán `nxt[i] = j`, `cnt[i] = done`.
3. Vòng `rows` lần: cộng `cnt[i]` rồi chuyển `i = nxt[i]`.

In [14]:
def build_transitions(sentence, cols):
    n = len(sentence)
    nxt = [0] * n
    cnt = [0] * n
    for i in range(n):
        j, remain, done = i, cols, 0
        while len(sentence[j]) <= remain:
            remain -= len(sentence[j]) + 1
            j += 1
            if j == n:
                j = 0
                done += 1
        nxt[i], cnt[i] = j, done
    return nxt, cnt


sentence, cols = ["a", "bcd", "e"], 6
print(build_transitions(sentence, cols))   # nxt = [2, 0, 1], cnt = [0, 1, 1]: hãy đối chiếu với mục 4

([2, 0, 1], [0, 1, 1])


Đối chiếu với tính tay ở mục 4: dòng bắt đầu bằng `a` (i=0) chứa `a bcd` và dừng trước `e` ⇒ `nxt[0] = 2` (từ `e`), `cnt[0] = 0`. Dòng bắt đầu bằng `e` (i=2) chứa `e a` rồi dừng trước `bcd` ⇒ `nxt[2] = 1`, `cnt[2] = 1` (đã đi qua cuối câu). Dòng bắt đầu bằng `bcd` (i=1) chứa `bcd e` ⇒ `nxt[1] = 0`, `cnt[1] = 1` (kết thúc câu và vừa hết dòng, từ kế là `a`).

In [15]:
def words_typing_dp(sentence, rows, cols):
    if any(len(w) > cols for w in sentence):
        return 0
    nxt, cnt = build_transitions(sentence, cols)
    i = ans = 0
    for _ in range(rows):
        ans += cnt[i]
        i = nxt[i]
    return ans


for sentence, rows, cols, expected in examples:
    assert words_typing_dp(sentence, rows, cols) == expected
print("Cách 3 đúng trên các ví dụ đề")

Cách 3 đúng trên các ví dụ đề


In [16]:
# Code nộp LeetCode — Cách 3
class Solution3:
    def wordsTyping(self, sentence: List[str], rows: int, cols: int) -> int:
        if any(len(w) > cols for w in sentence):
            return 0
        n = len(sentence)
        nxt = [0] * n
        cnt = [0] * n
        for i in range(n):
            j, remain, done = i, cols, 0
            while len(sentence[j]) <= remain:
                remain -= len(sentence[j]) + 1
                j += 1
                if j == n:
                    j = 0
                    done += 1
            nxt[i], cnt[i] = j, done
        i = ans = 0
        for _ in range(rows):
            ans += cnt[i]
            i = nxt[i]
        return ans


print(Solution3().wordsTyping(["i", "had", "apple", "pie"], 4, 5))   # 1

1


**Độ phức tạp:** tiền xử lý `n` dòng mô phỏng, mỗi dòng tối đa `cols / 2 + 1` từ ⇒ `O(n * cols)`; truy vấn `O(rows)`. Tổng `O(n * cols + rows)`, Space `O(n)`.

### Cách 4 — Phát hiện chu kỳ (cycle detection) để nhảy cóc `rows`

**Ý tưởng:** dãy trạng thái `i_0 = 0, i_1 = nxt[0], i_2 = nxt[i_1], ...` nằm trong tập `n` phần tử nên **chắc chắn lặp lại sau tối đa `n` dòng**. Khi gặp lại một trạng thái đã thấy ở dòng `r0` (với tổng `ans0` lúc đó), ta biết cứ mỗi `L = r - r0` dòng, tổng tăng thêm `gain = ans - ans0`. Ta nhảy cóc qua nhiều chu kỳ cùng lúc rồi mô phỏng nốt phần dư.

**Giải thích từng đoạn:**

1. `seen = {}` lưu `seen[i] = (r, ans)` — trạng thái `i` xuất hiện lần đầu ở dòng `r` với tổng `ans`.
2. Lặp `r` từ `0`: nếu `i in seen` ⇒ tính `cycle_len`, `cycle_gain`, `remaining = rows - r`, cộng `(remaining // cycle_len) * cycle_gain`, đặt `rows_left = remaining % cycle_len`, rồi mô phỏng nốt `rows_left` dòng và thoát.
3. Nếu chưa gặp: ghi nhớ, cộng `cnt[i]`, chuyển `i = nxt[i]`.

Cách này đáng giá khi `rows` cực lớn (ví dụ `10^9`), vượt khỏi ràng buộc bài; ở bài này nó chủ yếu là bài học về **chu kỳ trong đồ thị hàm**.

In [17]:
def words_typing_cycle(sentence, rows, cols):
    if any(len(w) > cols for w in sentence):
        return 0
    nxt, cnt = build_transitions(sentence, cols)
    seen = {}
    i = ans = 0
    r = 0
    while r < rows:
        if i in seen:
            r0, ans0 = seen[i]
            cycle_len, cycle_gain = r - r0, ans - ans0
            remaining = rows - r
            ans += (remaining // cycle_len) * cycle_gain     # nhảy cóc nhiều chu kỳ
            for _ in range(remaining % cycle_len):           # phần dư < cycle_len
                ans += cnt[i]
                i = nxt[i]
            return ans
        seen[i] = (r, ans)
        ans += cnt[i]
        i = nxt[i]
        r += 1
    return ans


for sentence, rows, cols, expected in examples:
    assert words_typing_cycle(sentence, rows, cols) == expected
print("Cách 4 đúng trên các ví dụ đề")
print("rows rất lớn:", words_typing_cycle(["a", "bcd", "e"], 10**12, 6))

Cách 4 đúng trên các ví dụ đề
rows rất lớn: 666666666666


In [18]:
# Code nộp LeetCode — Cách 4
class Solution4:
    def wordsTyping(self, sentence: List[str], rows: int, cols: int) -> int:
        if any(len(w) > cols for w in sentence):
            return 0
        n = len(sentence)
        nxt = [0] * n
        cnt = [0] * n
        for i in range(n):
            j, remain, done = i, cols, 0
            while len(sentence[j]) <= remain:
                remain -= len(sentence[j]) + 1
                j += 1
                if j == n:
                    j = 0
                    done += 1
            nxt[i], cnt[i] = j, done
        seen = {}
        i = ans = r = 0
        while r < rows:
            if i in seen:
                r0, ans0 = seen[i]
                length, gain = r - r0, ans - ans0
                left = rows - r
                ans += (left // length) * gain
                for _ in range(left % length):
                    ans += cnt[i]
                    i = nxt[i]
                return ans
            seen[i] = (r, ans)
            ans += cnt[i]
            i = nxt[i]
            r += 1
        return ans


print(Solution4().wordsTyping(["hello", "world"], 2, 8))   # 1

1


**Độ phức tạp:** tiền xử lý `O(n * cols)`; phần chạy `O(n)` (tối đa `n` dòng trước khi lặp + dư `< n`). Space `O(n)`.

**Bốn cách tóm tắt:** (1) mô phỏng từng từ; (2) một chuỗi vòng + lùi về dấu cách; (3) DP bảng chuyển trạng thái "từ bắt đầu dòng"; (4) chu kỳ trên đồ thị hàm `i -> nxt[i]`. Mỗi cách khai thác một góc nhìn khác nhau của cùng bài toán.

## 8. TRACE từng bước

Ta trace **Cách 2** (chuỗi vòng) trên ví dụ 3: `sentence = ["i","had","apple","pie"], rows = 4, cols = 5`. Chuỗi `s = "i had apple pie "` có `L = 16`.

Bảng trạng thái (hãy tự dự đoán từng dòng trước khi chạy code bên dưới, rồi so sánh):

| dòng | cur trước | cur + cols | ký tự tại `(cur+cols) % L` | hành động | cur sau |
|---|---|---|---|---|---|
| 1 | 0 | 5 | dấu cách (sau `had`) | `+1` | 6 |
| 2 | 6 | 11 | dấu cách (sau `apple`) | `+1` | 12 |
| 3 | 12 | 17 | dấu cách (sau `i`, đã quay vòng: `17 % 16 = 1`) | `+1` | 18 |
| 4 | 18 | 23 | chữ `p` (giữa `pie`) | lùi 1 bước | 22 |

Cột "hành động" là `+1` (rơi vào dấu cách) hoặc `lùi` (rơi giữa từ). Đáp án cuối: `22 // 16 = 1` câu trọn vẹn (dư `6`: đã viết thêm `i had ` của câu thứ hai).

In [19]:
def trace_joined(sentence, rows, cols):
    s = " ".join(sentence) + " "
    L = len(s)
    print(f"s = {s!r}, L = {L}, cols = {cols}")
    cur = 0
    for row in range(1, rows + 1):
        before = cur
        cur += cols
        ch = s[cur % L]
        if ch == " ":
            cur += 1
            action = "rơi vào dấu cách -> cur += 1"
        else:
            steps = 0
            while cur > 0 and s[(cur - 1) % L] != " ":
                cur -= 1
                steps += 1
            action = f"rơi giữa từ ({ch!r}) -> lùi {steps} bước"
        # phần chuỗi vô hạn đã phủ trong dòng này
        covered = "".join(s[p % L] for p in range(before, cur))
        print(f"dòng {row}: cur {before:>2} -> nhảy {before + cols:>2}, ký tự {ch!r:>4}, {action}; cur = {cur:>2}; dòng chứa {covered!r}")
    print("đáp án = cur // L =", cur // L, "(dư", cur % L, ")")
    return cur // L


assert trace_joined(["i", "had", "apple", "pie"], 4, 5) == 1

s = 'i had apple pie ', L = 16, cols = 5
dòng 1: cur  0 -> nhảy  5, ký tự  ' ', rơi vào dấu cách -> cur += 1; cur =  6; dòng chứa 'i had '
dòng 2: cur  6 -> nhảy 11, ký tự  ' ', rơi vào dấu cách -> cur += 1; cur = 12; dòng chứa 'apple '
dòng 3: cur 12 -> nhảy 17, ký tự  ' ', rơi vào dấu cách -> cur += 1; cur = 18; dòng chứa 'pie i '
dòng 4: cur 18 -> nhảy 23, ký tự  'p', rơi giữa từ ('p') -> lùi 1 bước; cur = 22; dòng chứa 'had '
đáp án = cur // L = 1 (dư 6 )


Đọc output: dòng 1 phủ `'i had '` (5 ký tự chữ + dấu cách cuối dòng), dòng 2 phủ `'apple '`, dòng 3 phủ `'pie i '` (đã sang đầu câu thứ hai), dòng 4: `cur + cols = 23` rơi vào chữ `p` của `pie` nên phải lùi 1 bước về đầu từ `had`, dòng 4 chỉ chứa `'had '`. Khớp màn hình ở đề: `i-had`, `apple`, `pie-i`, `had--`.

Ta cũng trace **Cách 3** (bảng chuyển trạng thái) trên ví dụ 2 để thấy sự khác biệt: ở đó ta không đụng tới ký tự, chỉ tra `nxt` và `cnt`.

In [20]:
def trace_dp(sentence, rows, cols):
    nxt, cnt = build_transitions(sentence, cols)
    print("nxt =", nxt, "| cnt =", cnt)
    i = ans = 0
    for row in range(1, rows + 1):
        print(f"dòng {row}: bắt đầu từ từ {i} ({sentence[i]!r}), cộng cnt[{i}] = {cnt[i]}, sang từ {nxt[i]}")
        ans += cnt[i]
        i = nxt[i]
        print(f"         tổng câu = {ans}")
    return ans


assert trace_dp(["a", "bcd", "e"], 3, 6) == 2

nxt = [2, 0, 1] | cnt = [0, 1, 1]
dòng 1: bắt đầu từ từ 0 ('a'), cộng cnt[0] = 0, sang từ 2
         tổng câu = 0
dòng 2: bắt đầu từ từ 2 ('e'), cộng cnt[2] = 1, sang từ 1
         tổng câu = 1
dòng 3: bắt đầu từ từ 1 ('bcd'), cộng cnt[1] = 1, sang từ 0
         tổng câu = 2


In [21]:
def trace_cycle(sentence, rows, cols):
    nxt, cnt = build_transitions(sentence, cols)
    seen = {}
    i = ans = r = 0
    while r < rows:
        print(f"r={r}: i={i}, ans={ans}, seen={seen}")
        if i in seen:
            r0, ans0 = seen[i]
            print(f"  -> trạng thái {i} đã gặp ở dòng {r0}: chu kỳ dài {r - r0}, mỗi chu kỳ thêm {ans - ans0} câu")
            length, gain, left = r - r0, ans - ans0, rows - r
            ans += (left // length) * gain
            print(f"  -> nhảy cóc {left // length} chu kỳ, còn dư {left % length} dòng")
            for _ in range(left % length):
                ans += cnt[i]
                i = nxt[i]
            return ans
        seen[i] = (r, ans)
        ans += cnt[i]
        i = nxt[i]
        r += 1
    return ans


print("kết quả:", trace_cycle(["a", "bcd", "e"], 100, 6))

r=0: i=0, ans=0, seen={}
r=1: i=2, ans=0, seen={0: (0, 0)}
r=2: i=1, ans=1, seen={0: (0, 0), 2: (1, 0)}
r=3: i=0, ans=2, seen={0: (0, 0), 2: (1, 0), 1: (2, 1)}
  -> trạng thái 0 đã gặp ở dòng 0: chu kỳ dài 3, mỗi chu kỳ thêm 2 câu
  -> nhảy cóc 32 chu kỳ, còn dư 1 dòng
kết quả: 66


## 9. Code nộp LeetCode

Bản được khuyến nghị: **Cách 2** (ngắn, dễ nhớ, đủ nhanh vì `rows <= 2 * 10^4` và mỗi dòng lùi tối đa 10 bước). Có kiểm tra thêm trường hợp từ dài hơn `cols` vẫn đúng (kết quả `0`) mà không cần nhánh riêng — ta sẽ chứng minh ở mục 10.

In [22]:
class Solution:
    def wordsTyping(self, sentence: List[str], rows: int, cols: int) -> int:
        s = " ".join(sentence) + " "     # có dấu cách ở cuối: mỗi từ đều "đi kèm" một dấu cách
        L = len(s)
        cur = 0                           # số ký tự của chuỗi vô hạn đã phủ xong
        for _ in range(rows):
            cur += cols                   # cố viết thêm cols ký tự
            if s[cur % L] == " ":         # rơi đúng vào dấu cách: dòng kết thúc ở ranh giới từ
                cur += 1                  # bỏ qua dấu cách (ô trống cuối dòng)
            else:                         # rơi giữa từ: đẩy cả từ xuống dòng sau
                while cur > 0 and s[(cur - 1) % L] != " ":
                    cur -= 1
        return cur // L


for sentence, rows, cols, expected in examples:
    got = Solution().wordsTyping(sentence, rows, cols)
    print(got, expected)
    assert got == expected
print(Solution().wordsTyping(["abcde", "f"], 5, 3))     # 0

1 1
2 2
1 1
0


## 10. Vì sao đúng

**Bất biến (invariant).** Sau mỗi dòng, `cur` là vị trí trong chuỗi vô hạn `s s s ...` sao cho:

1. Mọi ký tự trước `cur` đã được "đánh máy" xong (từ và dấu cách giữa các từ);
2. `cur` luôn đứng ở **đầu một từ** (nghĩa là `cur == 0` hoặc ký tự ngay trước `cur` là dấu cách).

**Bước chuyển.** Dòng mới bắt đầu tại đầu một từ ở vị trí `cur`. Nếu ta đánh máy liên tục (không bị cắt) thì dòng chứa các ký tự `[cur, cur + cols)`. Có hai khả năng:

- Ký tự thứ `cur + cols` (vị trí ngay sau cửa sổ) là **dấu cách** ⇒ cửa sổ `[cur, cur + cols)` kết thúc đúng cuối một từ ⇒ dòng này chứa trọn các từ; dấu cách đó là ô trống cuối dòng (không cần viết), dòng sau bắt đầu ở `cur + cols + 1`, vẫn là đầu một từ.
- Ký tự đó là **chữ** ⇒ từ chứa nó bị cắt bởi biên dòng ⇒ theo luật phải chuyển nguyên từ đó xuống dòng sau, tức dòng này chỉ gồm các từ trước nó. Vị trí bắt đầu dòng sau là đầu từ bị cắt ⇒ lùi `cur + cols` về đầu từ gần nhất (vị trí ngay sau dấu cách gần nhất ở phía trước). Đó đúng là cách tham lam: dòng chứa **nhiều từ nhất có thể**, và việc nhét nhiều từ nhất vào một dòng không bao giờ làm xấu đi các dòng sau (mô phỏng theo luật đề là xác định, không có lựa chọn).

**Vì sao kết quả là `cur // L`?** Mỗi lần `cur` vượt thêm `L` ký tự nghĩa là một câu trọn vẹn (kèm dấu cách cuối) đã được phủ. Vì `cur` luôn ở đầu từ nên phần dư `cur % L` là đầu một từ của câu hiện tại, không có "câu dở dang" nào bị tính nhầm.

**Từ dài hơn `cols`.** Nếu `cur` đang ở đầu một từ dài `m > cols` thì `cur + cols` rơi **giữa từ đó** (vì `cur + cols < cur + m`) ⇒ lùi về chính `cur` (ký tự trước `cur` là dấu cách) ⇒ `cur` không tăng. Từ đó trở đi `cur` kẹt mãi ⇒ đáp án là số câu đã hoàn thành trước đó. Nhưng vì từ dài hơn `cols` nằm trong câu nên câu không thể hoàn thành lần nào kể từ khi gặp nó, và `cur` chỉ có thể đến được từ đó sau khi hoàn thành một số từ đầu câu, nên `cur // L = 0` (chưa qua hết câu đầu tiên). Vậy kết quả `0` đúng.

**Vì sao `cur > 0` trong `while`?** Ở `cur = 0` không có ký tự trước; ta dừng. Điều này chỉ xảy ra ở dòng đầu khi từ đầu tiên dài hơn `cols`.

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng từng từ | `O(rows * cols)` (xấu nhất) | `O(1)` |
| 2. Chuỗi vòng + lùi | `O(rows * m)`, `m <= 10` | `O(L)`, `L <= 1100` |
| 3. Bảng `nxt/cnt` | `O(n * cols + rows)` | `O(n)` |
| 4. Chu kỳ | `O(n * cols)` | `O(n)` |

Với `n <= 100`, `rows, cols <= 2 * 10^4`: Cách 2 làm khoảng `2 * 10^4 * 10 = 2 * 10^5` thao tác — cực nhanh. Hãy đo thời gian thực để thấy khoảng cách giữa Cách 1 và các cách còn lại.

In [23]:
def solve2(sentence, rows, cols):
    return Solution().wordsTyping(sentence, rows, cols)


sentence = ["a"] * 100
rows = cols = 3000
for name, f in [("Cách 1 brute", words_typing_brute), ("Cách 2 joined", words_typing_joined),
                ("Cách 3 dp", words_typing_dp), ("Cách 4 cycle", words_typing_cycle)]:
    t = time.perf_counter()
    r = f(sentence, rows, cols)
    print(f"{name:<14}: kết quả {r}, {time.perf_counter() - t:.4f} giây")

Cách 1 brute  : kết quả 45000, 0.3006 giây
Cách 2 joined : kết quả 45000, 0.0003 giây
Cách 3 dp     : kết quả 45000, 0.0104 giây
Cách 4 cycle  : kết quả 45000, 0.0099 giây


## 12. Bộ test

Ta cần một **oracle độc lập**: hàm `render_screen` thật sự *vẽ* màn hình (danh sách chuỗi), rồi đếm số từ trên màn hình / số từ của câu. Oracle này không dùng dấu cách cuối hay chuỗi vòng nào.

In [24]:
def render_screen(sentence, rows, cols):
    """Vẽ màn hình bằng cách xếp từng từ theo luật đề, trả về (danh sách dòng, số câu trọn vẹn)."""
    n = len(sentence)
    if any(len(w) > cols for w in sentence):
        return [""] * rows, 0
    screen = []
    placed = 0                      # tổng số từ đã đặt
    for _ in range(rows):
        line = ""
        while True:
            w = sentence[placed % n]
            candidate = w if line == "" else line + " " + w
            if len(candidate) > cols:
                break
            line = candidate
            placed += 1
        screen.append(line)
    return screen, placed // n


screen, ans = render_screen(["i", "had", "apple", "pie"], 4, 5)
for line in screen:
    print(line.ljust(5, "-"))
print("số câu:", ans)

i had
apple
pie i
had--
số câu: 1


In [25]:
# 12.1. Ví dụ của đề cho tất cả các cách
solutions = {
    "brute": words_typing_brute,
    "joined": words_typing_joined,
    "dp": words_typing_dp,
    "cycle": words_typing_cycle,
    "LeetCode class": lambda s, r, c: Solution().wordsTyping(s, r, c),
}
for sentence, rows, cols, expected in examples:
    for name, f in solutions.items():
        assert f(sentence, rows, cols) == expected, (name, sentence, rows, cols)
print("12.1 OK: mọi cách đúng ví dụ đề")

12.1 OK: mọi cách đúng ví dụ đề


In [26]:
# 12.2. Edge case
edge = [
    (["a"], 1, 1, 1),                       # vừa khít một ô
    (["a"], 1, 2, 1),                       # "a" + 1 ô trống; "a a" cần 3 ô
    (["a"], 1, 3, 2),                       # "a a"
    (["a"], 3, 1, 3),                       # mỗi dòng một chữ
    (["abcde", "f"], 5, 3, 0),              # từ dài hơn cols
    (["abcde", "f"], 5, 5, 2),              # dòng: abcde / f -> mỗi 2 dòng một câu
    (["ab", "cd"], 1, 4, 0),                # "ab c" không vừa: dòng 1 = "ab" (thiếu "cd")
    (["ab", "cd"], 1, 5, 1),                # "ab cd" vừa khít
    (["x" * 10] * 100, 20000, 20000, None), # max, chỉ kiểm tra không lỗi và các cách khớp nhau
]
for sentence, rows, cols, expected in edge:
    outs = {name: f(sentence, rows, cols) for name, f in solutions.items() if name != "brute" or rows * cols <= 10**6}
    vals = set(outs.values())
    assert len(vals) == 1, (sentence[:2], rows, cols, outs)
    got = vals.pop()
    if expected is not None:
        assert got == expected, (sentence, rows, cols, got, expected)
        assert render_screen(sentence, rows, cols)[1] == expected
    print(f"{sentence[:2]}... rows={rows} cols={cols} -> {got}")
print("12.2 OK")

['a']... rows=1 cols=1 -> 1
['a']... rows=1 cols=2 -> 1
['a']... rows=1 cols=3 -> 2
['a']... rows=3 cols=1 -> 3
['abcde', 'f']... rows=5 cols=3 -> 0
['abcde', 'f']... rows=5 cols=5 -> 2
['ab', 'cd']... rows=1 cols=4 -> 0
['ab', 'cd']... rows=1 cols=5 -> 1
['xxxxxxxxxx', 'xxxxxxxxxx']... rows=20000 cols=20000 -> 363600
12.2 OK


In [27]:
# 12.3. Ngẫu nhiên có seed, đối chiếu với oracle độc lập
random.seed(418)
for _ in range(3000):
    n = random.randint(1, 6)
    sentence = ["".join(random.choice("abc") for _ in range(random.randint(1, 5))) for _ in range(n)]
    rows = random.randint(1, 12)
    cols = random.randint(1, 14)
    expected = render_screen(sentence, rows, cols)[1]
    for name, f in solutions.items():
        got = f(sentence, rows, cols)
        assert got == expected, (name, sentence, rows, cols, got, expected)
print("12.3 OK: 3000 test ngẫu nhiên khớp oracle ở cả 5 cách")

12.3 OK: 3000 test ngẫu nhiên khớp oracle ở cả 5 cách


In [28]:
# 12.4. Test kích thước lớn: các cách nhanh khớp nhau (không dùng oracle/brute chậm)
random.seed(7)
for _ in range(30):
    n = random.randint(1, 100)
    sentence = ["".join(random.choice("abcdefghij") for _ in range(random.randint(1, 10))) for _ in range(n)]
    rows = random.randint(1, 20000)
    cols = random.randint(1, 20000)
    a = words_typing_joined(sentence, rows, cols)
    b = words_typing_dp(sentence, rows, cols)
    c = words_typing_cycle(sentence, rows, cols)
    assert a == b == c, (a, b, c)
print("12.4 OK: 30 test lớn, ba cách nhanh khớp nhau")

12.4 OK: 30 test lớn, ba cách nhanh khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên dấu cách ở cuối câu

Viết `s = " ".join(sentence)` (không cộng `" "`) thì mối nối giữa hai câu liên tiếp bị thiếu một dấu cách, và công thức `cur // len(s)` đếm sai độ dài câu. Hãy dùng oracle để tìm phản ví dụ nhỏ nhất.

In [29]:
def wrong_no_trailing_space(sentence, rows, cols):
    s = " ".join(sentence)               # SAI: thiếu dấu cách cuối
    L = len(s)
    cur = 0
    for _ in range(rows):
        cur += cols
        if s[cur % L] == " ":
            cur += 1
        else:
            while cur > 0 and s[(cur - 1) % L] != " ":
                cur -= 1
    return cur // L


random.seed(1)
found = None
for _ in range(10000):
    sentence = ["".join(random.choice("ab") for _ in range(random.randint(1, 3))) for _ in range(random.randint(1, 3))]
    rows, cols = random.randint(1, 5), random.randint(1, 7)
    if wrong_no_trailing_space(sentence, rows, cols) != render_screen(sentence, rows, cols)[1]:
        found = (sentence, rows, cols)
        break
print("phản ví dụ:", found)
print("sai  :", wrong_no_trailing_space(*found))
print("đúng :", render_screen(*found)[1])

phản ví dụ: (['aba'], 4, 7)
sai  : 0
đúng : 8


**Vì sao sai?** Không có dấu cách cuối thì "câu" bị đo thiếu 1 ký tự, trong khi trên màn hình luôn cần một ô cách giữa từ cuối và từ đầu của câu kế. Sửa: `s = " ".join(sentence) + " "`.

### Lỗi 2 — Kiểm tra ký tự `s[cur]` thay vì `s[cur - 1]` khi lùi

In [30]:
def wrong_off_by_one(sentence, rows, cols):
    s = " ".join(sentence) + " "
    L = len(s)
    cur = 0
    for _ in range(rows):
        cur += cols
        if s[cur % L] == " ":
            cur += 1
        else:
            while cur > 0 and s[cur % L] != " ":      # SAI: phải xét ký tự TRƯỚC cur
                cur -= 1
    return cur // L


random.seed(2)
found = None
for _ in range(10000):
    sentence = ["".join(random.choice("ab") for _ in range(random.randint(1, 3))) for _ in range(random.randint(1, 3))]
    rows, cols = random.randint(1, 5), random.randint(1, 7)
    if wrong_off_by_one(sentence, rows, cols) != render_screen(sentence, rows, cols)[1]:
        found = (sentence, rows, cols)
        break
print("phản ví dụ:", found)
print("sai  :", wrong_off_by_one(*found))
print("đúng :", render_screen(*found)[1])

phản ví dụ: (['abb', 'bbb', 'a'], 3, 4)
sai  : 0
đúng : 1


**Vì sao sai?** Khi lùi tới dấu cách tại vị trí `cur` thì `cur` đang **ở chính dấu cách** chứ không phải ở đầu từ kế tiếp; dòng sau sẽ bắt đầu bằng một dấu cách thừa. Ta muốn `cur` đứng **ngay sau** dấu cách ⇒ kiểm tra `s[cur - 1]`.

### Lỗi 3 — Brute force không chặn từ dài hơn `cols` (vòng lặp vô hạn)

Nếu từ `sentence[i]` dài hơn `cols`, vòng `while len(sentence[i]) <= remain` không bao giờ chạy, `i` không nhích, và nếu bạn viết kiểu "lặp tới khi hết ô" bằng `while True` thì sẽ treo. Ở đây ta mô phỏng treo bằng bộ đếm bước để notebook không bị đứng.

In [31]:
def wrong_brute_may_hang(sentence, rows, cols, limit=1000):
    n = len(sentence)
    i = count = steps = 0
    for _ in range(rows):
        remain = cols
        while True:                                  # SAI: lặp cho tới khi "tự dừng"
            steps += 1
            if steps > limit:
                return "TREO (vòng lặp không dừng)"
            if len(sentence[i]) > remain:
                if remain == cols:                   # dòng trống mà từ vẫn không vừa -> kẹt vĩnh viễn
                    continue
                break
            remain -= len(sentence[i]) + 1
            i += 1
            if i == n:
                i, count = 0, count + 1
    return count


print(wrong_brute_may_hang(["abcde", "f"], 3, 3))    # từ 'abcde' dài hơn cols
print(words_typing_brute(["abcde", "f"], 3, 3))      # bản đúng: chặn sớm, trả 0

TREO (vòng lặp không dừng)
0


**Sửa:** kiểm tra `any(len(w) > cols for w in sentence)` ngay đầu hàm và trả `0`.

### Lỗi 4 — Đếm `cur // L` nhưng tăng `cur` sai khi dòng kết thúc tại dấu cách

Nếu quên `cur += 1` khi `s[cur % L] == " "` thì dòng sau bắt đầu bằng dấu cách và bị "trừ hao" một ô, làm một số dòng chứa ít từ hơn thực tế.

In [32]:
def wrong_forget_plus_one(sentence, rows, cols):
    s = " ".join(sentence) + " "
    L = len(s)
    cur = 0
    for _ in range(rows):
        cur += cols
        if s[cur % L] != " ":                        # SAI: nhánh dấu cách không làm gì
            while cur > 0 and s[(cur - 1) % L] != " ":
                cur -= 1
    return cur // L


random.seed(3)
found = None
for _ in range(10000):
    sentence = ["".join(random.choice("ab") for _ in range(random.randint(1, 3))) for _ in range(random.randint(1, 3))]
    rows, cols = random.randint(1, 5), random.randint(1, 7)
    if wrong_forget_plus_one(sentence, rows, cols) != render_screen(sentence, rows, cols)[1]:
        found = (sentence, rows, cols)
        break
print("phản ví dụ:", found, "| sai:", wrong_forget_plus_one(*found), "| đúng:", render_screen(*found)[1])

phản ví dụ: (['bb'], 4, 5) | sai: 6 | đúng: 8


**Bài học chung:** với bài mô phỏng có nhiều điều kiện biên, **luôn viết oracle độc lập đơn giản** rồi để máy tìm phản ví dụ nhỏ nhất — nhanh hơn nhiều so với suy luận bằng mắt.

## 14. Mẹo & nhận dạng pattern

- **"Lặp lại một cấu trúc vô hạn rồi cắt theo độ dài cố định"** ⇒ nghĩ ngay tới chuỗi vòng và `%`, `//`.
- **"Mỗi bước chỉ phụ thuộc vào trạng thái nhỏ (từ bắt đầu dòng)"** ⇒ lập bảng chuyển trạng thái (DP/functional graph), rồi tận dụng chu kỳ.
- **Nhúng dấu cách vào chuỗi** (dấu cách cuối câu) biến luật "có dấu cách giữa các từ" thành luật đồng nhất, giảm số trường hợp.
- **"Không được cắt đôi từ"** ⇒ lùi về ranh giới gần nhất (giống tách dòng/word wrap).
- Với bài có `rows * cols` rất lớn nhưng số trạng thái nhỏ, đừng bao giờ mô phỏng từng ô.
- Cách tự kiểm tra: viết hàm vẽ màn hình để *nhìn* kết quả như đề.

## 15. Biến thể

1. **Số dấu cách giữa các từ là `g` (thay vì 1):** thay `" "` bằng `" " * g` hoặc tăng chi phí `len(word) + g` trong Cách 1/3. Chuỗi vòng vẫn dùng được nếu coi `g` dấu cách là một khối (nhưng phải lùi theo khối).
2. **Chiều rộng từng dòng khác nhau:** chuỗi vòng không còn thích hợp vì bước nhảy không cố định; bảng `nxt/cnt` theo từng dòng vẫn dùng được.
3. **Đếm số từ (không phải số câu):** trả `cur` đã điều chỉnh, hoặc đếm tổng số từ được đặt trong Cách 1/3.
4. **Căn giữa/căn đều (text justification, LeetCode 68):** cùng nền word-wrap tham lam, nhưng phải phân phối dấu cách thêm.
5. **Số dòng tối thiểu để viết `k` câu:** đảo ngược bài toán — lặp tới khi `cur // L >= k` (hoặc chia nhị phân trên `rows`, vì đáp án đơn điệu theo `rows`).

In [33]:
# Biến thể 5: số dòng tối thiểu để viết ít nhất k câu (mô phỏng bằng Cách 2 mở rộng)
def min_rows_for(sentence, cols, k):
    if any(len(w) > cols for w in sentence):
        return -1
    s = " ".join(sentence) + " "
    L = len(s)
    cur = rows = 0
    while cur // L < k:
        cur += cols
        if s[cur % L] == " ":
            cur += 1
        else:
            while cur > 0 and s[(cur - 1) % L] != " ":
                cur -= 1
        rows += 1
    return rows


print(min_rows_for(["a", "bcd", "e"], 6, 2))   # 3, khớp ví dụ 2
assert words_typing_joined(["a", "bcd", "e"], 3, 6) >= 2
assert words_typing_joined(["a", "bcd", "e"], 2, 6) < 2

3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng từng từ | Đặt lần lượt từng từ, trừ ô còn lại | `O(rows * cols)` xấu nhất | `O(1)` | Hiểu đề, làm oracle trên input nhỏ |
| 2. Chuỗi vòng + lùi | Nối câu + dấu cách, nhảy `cols` mỗi dòng | `O(rows * m)` | `O(L)` | **Khuyên dùng khi thi**: ngắn, nhanh |
| 3. Bảng chuyển trạng thái | Tính trước dòng bắt đầu từ `i` | `O(n * cols + rows)` | `O(n)` | Khi cần tái sử dụng bảng, nhiều truy vấn `rows` |
| 4. Phát hiện chu kỳ | Đồ thị hàm `i -> nxt[i]` lặp tuần hoàn | `O(n * cols)` | `O(n)` | `rows` cực lớn (`10^9+`) |

Cả bốn cách cho cùng đáp án trên mọi test ở mục 12.

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `len(" ".join(["ab", "c"]) + " ")`; (b) `"ab c "[8 % 5]`; (c) `17 // 8` và `17 % 8`; (d) có bao nhiêu câu trọn vẹn nếu đã phủ `cur = 24` ký tự của chuỗi dài `L = 8`.

**Bài 2 (tính tay).** Với `sentence = ["ab", "cd"]`, `rows = 3`, `cols = 5`, vẽ màn hình (dùng `-` cho ô trống) và đếm số câu trọn vẹn. Sau đó tính lại bằng chuỗi vòng: `s = "ab cd "`.

**Bài 3 (điền TODO).** Hoàn thành `words_typing_todo(sentence, rows, cols)` theo Cách 2.

In [34]:
def words_typing_todo(sentence, rows, cols):
    # TODO 1: s = " ".join(sentence) + " " ; L = len(s) ; cur = 0
    # TODO 2: lặp rows lần: cur += cols
    # TODO 3: nếu s[cur % L] == " " thì cur += 1
    # TODO 4: ngược lại lùi cur trong khi cur > 0 và s[(cur - 1) % L] != " "
    # TODO 5: trả về cur // L
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `words_typing_dp2(sentence, rows, cols)` theo Cách 3 **không nhìn lại** code ở mục 7. Kiểm tra với ví dụ 1, 2, 3 của đề.

**Bài 5 (biến thể / mini challenge).** Viết `total_words(sentence, rows, cols)` trả về **tổng số từ** (không phải số câu) được viết lên màn hình, kể cả phần câu dở dang cuối màn hình. Kiểm tra: `["a","bcd","e"], 3, 6 → 6` (màn hình `a bcd`, `e a`, `bcd e` có 2 + 2 + 2 = 6 từ).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `5`; `8 % 5 = 3` rồi tra chỉ số 3 của `"ab c "` được `'c'`; `2` và `1`; `24 // 8 = 3`. Bài 3: nhìn lại hàm `words_typing_joined` ở mục 7.
- **Mức 2 (vừa):** Bài 2: dòng 1 `ab cd` (5 ô, vừa khít) → dòng 2 `ab cd` → dòng 3 `ab cd` ⇒ 3 câu. Bài 4: tiền xử lý `nxt[i], cnt[i]` rồi lặp `rows` lần tra bảng.
- **Mức 3 (sâu):** Bài 5: dùng oracle `render_screen` để đếm số từ đã đặt: viết lại vòng đặt từ nhưng trả về `placed` thay vì `placed // n`; hoặc từ bảng `nxt/cnt` thêm mảng `words_in_row[i]` (số từ đặt được trong dòng bắt đầu từ `i`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [35]:
# Bài 1
assert len(" ".join(["ab", "c"]) + " ") == 5
assert "ab c "[8 % 5] == "c"
assert (17 // 8, 17 % 8) == (2, 1)
assert 24 // 8 == 3

# Bài 2
screen, ans = render_screen(["ab", "cd"], 3, 5)
assert screen == ["ab cd", "ab cd", "ab cd"] and ans == 3
assert words_typing_joined(["ab", "cd"], 3, 5) == 3


# Bài 3
def words_typing_todo_ans(sentence, rows, cols):
    s = " ".join(sentence) + " "
    L = len(s)
    cur = 0
    for _ in range(rows):
        cur += cols
        if s[cur % L] == " ":
            cur += 1
        else:
            while cur > 0 and s[(cur - 1) % L] != " ":
                cur -= 1
    return cur // L


for sentence, rows, cols, expected in examples:
    assert words_typing_todo_ans(sentence, rows, cols) == expected


# Bài 4
def words_typing_dp2(sentence, rows, cols):
    if any(len(w) > cols for w in sentence):
        return 0
    n = len(sentence)
    nxt, cnt = [0] * n, [0] * n
    for i in range(n):
        j, remain, done = i, cols, 0
        while len(sentence[j]) <= remain:
            remain -= len(sentence[j]) + 1
            j = (j + 1) % n
            if j == 0:
                done += 1
        nxt[i], cnt[i] = j, done
    i = ans = 0
    for _ in range(rows):
        ans += cnt[i]
        i = nxt[i]
    return ans


for sentence, rows, cols, expected in examples:
    assert words_typing_dp2(sentence, rows, cols) == expected


# Bài 5
def total_words(sentence, rows, cols):
    if any(len(w) > cols for w in sentence):
        return 0
    n = len(sentence)
    nxt, wcount = [0] * n, [0] * n
    for i in range(n):
        j, remain, w = i, cols, 0
        while len(sentence[j]) <= remain:
            remain -= len(sentence[j]) + 1
            j = (j + 1) % n
            w += 1
        nxt[i], wcount[i] = j, w
    i = total = 0
    for _ in range(rows):
        total += wcount[i]
        i = nxt[i]
    return total


def total_words_oracle(sentence, rows, cols):
    screen, _ = render_screen(sentence, rows, cols)
    return sum(len(line.split()) for line in screen)


print("total_words ví dụ 2:", total_words(["a", "bcd", "e"], 3, 6))
assert total_words(["a", "bcd", "e"], 3, 6) == total_words_oracle(["a", "bcd", "e"], 3, 6) == 6
random.seed(17)
for _ in range(500):
    sentence = ["".join(random.choice("ab") for _ in range(random.randint(1, 4))) for _ in range(random.randint(1, 5))]
    rows, cols = random.randint(1, 10), random.randint(1, 12)
    assert total_words(sentence, rows, cols) == total_words_oracle(sentence, rows, cols)
print("OK: toàn bộ đáp án đúng")

total_words ví dụ 2: 6
OK: toàn bộ đáp án đúng


Lưu ý về Bài 5: màn hình ví dụ 2 là `a bcd` / `e a` / `bcd e` nên có 6 từ (nhưng chỉ 2 câu trọn vẹn, vì 6 từ = 2 câu x 3 từ). Hai cách tính độc lập (bảng `wcount` và oracle vẽ màn hình) cho cùng kết quả, đó là cách tự kiểm tra tốt.

## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (chuỗi vòng + lùi về dấu cách)

```python
class Solution:
    def wordsTyping(self, sentence: List[str], rows: int, cols: int) -> int:
        s = " ".join(sentence) + " "          # dấu cách cuối câu
        L, cur = len(s), 0
        for _ in range(rows):
            cur += cols                        # nhảy một dòng
            if s[cur % L] == " ":
                cur += 1                       # kết thúc đúng ranh giới từ
            else:
                while cur > 0 and s[(cur - 1) % L] != " ":
                    cur -= 1                   # đẩy từ dở dang xuống dòng sau
        return cur // L
```

**Checklist**

- [ ] Đã nối `" ".join(sentence) + " "` (có dấu cách **cuối**)?
- [ ] Điều kiện lùi xét `s[(cur - 1) % L]` (ký tự **trước** `cur`), không phải `s[cur % L]`?
- [ ] Nhánh rơi vào dấu cách có `cur += 1`?
- [ ] Đáp án là `cur // L` (không phải `cur // len(sentence)`)?
- [ ] Đã nghĩ tới từ dài hơn `cols` (đáp án `0`, không treo)?
- [ ] Đã so sánh với oracle vẽ màn hình trên input nhỏ ngẫu nhiên?
- [ ] Đề Premium: đã đối chiếu lại đề gốc trên LeetCode nếu có tài khoản?